In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

# src 경로 등록 + config 로드
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from bootstrap import setup_notebook
ROOT, cfg, _vars = setup_notebook("dog.yaml")
globals().update(_vars)

import os
import json
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import torch
from diffusers import StableDiffusionPipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
if "pipe" not in globals():
    pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")
    pipe = pipe.to(DEVICE)
    pipe.safety_checker = None
    pipe.set_progress_bar_config(disable=True)


In [ ]:
from load import load_cache
_cache = load_cache(cfg, "neuron_selection")
for k, v in _cache.items():
    globals()[k] = v
if hasattr(dino_embeds, 'numpy'):
    dino_embeds = dino_embeds.numpy()


# Appendix (supplementary)

## A. Population-level prevalence — 전체 5120 뉴런
> **(부록)** 본문은 선택된 coarse 분할에 집중하고, 이 분석은 보충 증거로 둔다.  
> **주장**: 선택된 몇 개 뉴런만의 얘기가 아니라, GEGLU gate의 **수많은 뉴런**이 통계적으로 유의미한 semantic 분할을 만든다 (cherry-pick 아님).  
> `dino_sim`을 한 번만 계산하고 모든 뉴런을 vectorized/chunk 방식으로 처리.

In [ ]:
# ── 랜덤 50:50 이진분할 기준선 (200회) — 단일뉴런 검증에서 이동 ──────────────
def binary_intra_inter(embeds, labels):
    """labels ∈ {0,1} → (intra, inter, gap)"""
    pos, neg = embeds[labels == 1], embeds[labels == 0]
    def mean_pairs(A, B=None):
        if B is None:
            if len(A) < 2: return np.nan
            sim = A @ A.T
            m = np.triu(np.ones(len(A), dtype=bool), k=1)
            return float(sim[m].mean())
        return float((A @ B.T).mean())
    intra = np.nanmean([mean_pairs(pos), mean_pairs(neg)])
    inter = mean_pairs(pos, neg)
    return intra, inter, intra - inter

_rng_base = np.random.default_rng(0)
rand_gaps = []
for _ in range(200):
    _lbl = _rng_base.permutation(np.array([0] * (N // 2) + [1] * (N - N // 2)))
    _, _, _g = binary_intra_inter(dino_embeds, _lbl)
    rand_gaps.append(_g)
rand_mean = float(np.mean(rand_gaps))
rand_std  = float(np.std(rand_gaps))
print(f"Random 50:50 baseline  gap = {rand_mean:.5f} ± {rand_std:.5f}")

import time

# ── 1) DINOv2 pairwise similarity — 한 번만 계산 ─────────────────────────────
dino_sim  = dino_embeds @ dino_embeds.T                   # (N, N)
triu_i, triu_k = np.triu_indices(N, k=1)
sim_pairs = dino_sim[triu_i, triu_k].astype(np.float32)  # (P,)  P = N*(N-1)/2

# binary (N, J) — 위 셀에서 이미 계산됨
J    = binary.shape[1]   # 5120
B_np = binary.numpy().astype(np.bool_)
B_i  = B_np[triu_i]     # (P, J)
B_k  = B_np[triu_k]     # (P, J)

# ── 2) Chunk별 gap 계산 (메모리 안전) ─────────────────────────────────────────
CHUNK     = 512
all_gaps  = np.zeros(J, dtype=np.float32)
valid_mask = np.zeros(J, dtype=bool)

t0 = time.time()
for start in range(0, J, CHUNK):
    end = min(start + CHUNK, J)
    Bi  = B_i[:, start:end]           # (P, chunk)
    Bk  = B_k[:, start:end]           # (P, chunk)

    intra_mask = (Bi & Bk) | (~Bi & ~Bk)   # (P, chunk)
    inter_mask = ~intra_mask

    s = sim_pairs[:, None]             # (P, 1)

    intra_sum = (intra_mask.astype(np.float32) * s).sum(axis=0)
    intra_cnt = intra_mask.sum(axis=0).astype(np.float32)
    inter_sum = (inter_mask.astype(np.float32) * s).sum(axis=0)
    inter_cnt = inter_mask.sum(axis=0).astype(np.float32)

    # 유효 조건: 각 그룹 ≥ 5개 (이미지 기준 양성/음성 개수)
    n_pos = B_np[:, start:end].sum(axis=0).astype(np.int32)
    n_neg = N - n_pos
    chunk_valid = (n_pos >= 5) & (n_neg >= 5)
    valid_mask[start:end] = chunk_valid

    gap = (np.where(intra_cnt > 0, intra_sum / intra_cnt, 0.0)
           - np.where(inter_cnt > 0, inter_sum / inter_cnt, 0.0))
    all_gaps[start:end] = gap

elapsed = time.time() - t0
print(f"계산 완료: {J}개 뉴런  ({elapsed:.1f}s)")

# ── 3) z-score 및 유의 뉴런 수 ─────────────────────────────────────────────────
# rand_mean, rand_std: 위에서 계산된 200회 random 기준선
valid_gaps = all_gaps[valid_mask]
n_valid    = int(valid_mask.sum())
z_all      = (valid_gaps - rand_mean) / rand_std

SIGMA_THRESH  = 2.0
n_sig         = int((z_all > SIGMA_THRESH).sum())
n_sig_pct     = n_sig / n_valid * 100
n_expected    = n_valid * 0.0228   # 정규분포 기준 2σ 초과 기대값 (2.28%)

print(f"유효 뉴런 (각 그룹 ≥5)  : {n_valid}/{J}")
print(f">2σ 유의 뉴런           : {n_sig}/{n_valid}  ({n_sig_pct:.1f}%)")
print(f"랜덤 기대값             : {n_expected:.0f}  (2.28%)")
print(f"실제/기대 배율          : {n_sig_pct/2.28:.1f}×")

# ── 4) 선택된 3개 뉴런의 전체 분포 내 위치 ────────────────────────────────────
print(f"\n선택된 뉴런의 전체 분포 내 순위:")
valid_idx = np.where(valid_mask)[0]
rank_total = np.argsort(-all_gaps[valid_mask])   # gap 내림차순
for j in sel:
    g  = float(all_gaps[j])
    z  = (g - rand_mean) / rand_std
    # valid_gaps에서의 순위
    rank = int((valid_gaps > g).sum()) + 1
    print(f"  neuron #{j:4d}: gap={g:.5f}  z={z:+.1f}σ  rank={rank}/{n_valid}")

# ── 5) Top-10 뉴런 ────────────────────────────────────────────────────────────
top10 = np.argsort(-valid_gaps)[:10]
print(f"\nTop-10 neurons (by DINOv2 coherence gap):")
for rank, idx in enumerate(top10, 1):
    j   = int(valid_idx[idx])
    g   = float(valid_gaps[idx])
    z   = float(z_all[idx])
    tag = " ← sel" if j in sel else ""
    print(f"  #{rank:2d}  neuron #{j:4d}: gap={g:.5f}  z={z:+.1f}σ{tag}")

In [ ]:
# ── Publication style (matches multi-plane figure) ───────────────────────────
FIG_BG  = '#FFFFFF'
GRID    = '#D5D9DE'
SPINE   = '#3D3D3D'
TEXT    = '#2A2A2A'
MUTED   = '#666666'
REFLINE = '#8A8A8A'

C_MAIN   = '#2F4B7C'   # slate blue (reference / threshold)
C_SIG    = '#B85450'   # muted brick — significant neurons
C_NULL   = '#A8A8A8'   # neutral gray — not significant / expected
C_RAND   = REFLINE
C_THRESH = '#2F4B7C'   # 2σ threshold (reference line)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica Neue', 'DejaVu Sans'],
    'font.size': 10,
    'axes.labelcolor': TEXT,
    'axes.edgecolor': SPINE,
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': GRID,
    'grid.alpha': 0.85,
    'grid.linewidth': 0.6,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
    'text.color': TEXT,
    'figure.facecolor': FIG_BG,
    'axes.facecolor': FIG_BG,
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
fig.subplots_adjust(wspace=0.38, bottom=0.18)

# ── 왼쪽: Ranked z-score plot ─────────────────────────────────────────────────
sorted_idx = np.argsort(z_all)
z_sorted   = z_all[sorted_idx]
ranks      = np.arange(1, n_valid + 1)
below      = z_sorted <= SIGMA_THRESH
above      = z_sorted >  SIGMA_THRESH

ax1.scatter(ranks[below], z_sorted[below],
            c=C_NULL, s=13, alpha=0.50, zorder=3, label='Not significant',
            edgecolors='none')
ax1.scatter(ranks[above], z_sorted[above],
            c=C_SIG, s=15, alpha=0.88, zorder=4,
            edgecolors='white', linewidths=0.4,
            label=f'> 2σ  —  {n_sig} planes  ({n_sig_pct:.1f}%)')

ax1.axhline(SIGMA_THRESH, color=C_THRESH, lw=1.3, ls=(0, (5, 4)), zorder=5,
            label='2σ threshold')
ax1.axhline(0, color=C_RAND, lw=0.9, ls=(0, (4, 4)), alpha=0.75, zorder=2,
            label='Random baseline  (z = 0)')

# 선택된 평면(뉴런) 강조 (별 + 라벨 겹침 방지: z 오름차순으로 라벨 높이 분리)
valid_idx_arr = np.where(valid_mask)[0]
_stars = []
for j in sel:
    matches = np.where(valid_idx_arr == j)[0]
    if len(matches) == 0:
        continue
    z_j    = float(z_all[matches[0]])
    rank_j = int((z_sorted < z_j).sum()) + 1
    _stars.append((rank_j, z_j, j))
_stars.sort(key=lambda t: t[1])
_min_gap = (z_all.max() - z_all.min()) * 0.09
_label_y = -np.inf

ax1.set_xlabel('plane rank  (sorted by z-score, low → high)', fontsize=9.5, color=TEXT)
ax1.set_ylabel('z-score  (vs. random baseline)', fontsize=9.5, color=TEXT)
ax1.legend(fontsize=8.3, framealpha=1.0, edgecolor=GRID, facecolor=FIG_BG, loc='upper left')
ax1.set_xlim(0, n_valid + 5)
# ax1.set_ylim(-5, z_all.max() * 1.05)
ax1.tick_params(axis='both', labelsize=9)

# ── 오른쪽: Expected vs Observed ─────────────────────────────────────────────
bar_x      = [0, 1]
bar_vals   = [n_expected, n_sig]
bar_colors = [C_NULL, C_SIG]

bars = ax2.bar(bar_x, bar_vals, color=bar_colors, width=0.45,
               alpha=0.92, edgecolor=SPINE, linewidth=0.7, zorder=3)

for bar, v, lbl in zip(bars, bar_vals,
                        [f'Expected\n(random)\n{n_expected:.0f} planes\n(2.28%)',
                         f'Observed\n(gate planes)\n{n_sig} planes\n({n_sig_pct:.1f}%)']):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + n_sig * 0.02,
             lbl, ha='center', va='bottom', fontsize=9.5, fontweight='600', color=TEXT)

_arr_start = (0.16, n_expected + n_sig * 0.06)
_arr_end   = (0.84, n_sig * 0.92)
mid_x = (_arr_start[0] + _arr_end[0]) / 2
mid_y = (_arr_start[1] + _arr_end[1]) / 2
ax2.text(mid_x, mid_y, f'≈ {n_sig_pct/2.28:.1f}×',
         ha='center', va='center', fontsize=14, fontweight='bold',
         color=C_SIG, zorder=6,
         bbox=dict(boxstyle='round,pad=0.28', fc=FIG_BG, ec=GRID, alpha=0.95, lw=0.8))

ax2.set_xticks([0, 1])
ax2.set_xticklabels(['Expected\n(random)', 'Observed\n(gate planes)'], fontsize=9.5)
ax2.set_ylabel('# Planes exceeding 2σ threshold', fontsize=9.5, color=TEXT)
ax2.set_ylim(0, n_sig * 1.4)
ax2.tick_params(axis='both', labelsize=9)

plt.savefig('results_population_ranked.png', dpi=160, bbox_inches='tight', facecolor=FIG_BG)
plt.show()
print("saved: results_population_ranked.png")

## B. 블록 × 프롬프트 재현성 (small-multiples 격자 + 요약 표)

각 `(block, prompt)` 설정을 실행한 **직후** 아래 `save_appendix_result(...)`를 한 번 호출해
`z_all` 분포를 `appendix_results/`에 누적 저장합니다. 그런 다음 마지막 셀을 실행하면
저장된 모든 설정으로 격자 figure와 요약 표가 자동 생성됩니다. (설정이 늘면 다시 실행만 하면 됨)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Appendix 결과 누적 저장: 각 block/prompt 설정 실행 직후 1회 호출
# ════════════════════════════════════════════════════════════════════════════
import os, numpy as np

APPENDIX_DIR = 'appendix_results'
os.makedirs(APPENDIX_DIR, exist_ok=True)

def save_appendix_result(block, prompt, z_all, sigma_thresh=2.0, n_total=None):
    """현재 실행한 설정의 z-score 분포를 .npz로 저장한다."""
    z   = np.asarray(z_all, dtype=float)
    key = f"{block}__{prompt}".lower().replace(' ', '_').replace('/', '-')
    payload = dict(block=block, prompt=prompt, z_all=z, sigma_thresh=float(sigma_thresh))
    if n_total is not None:
        payload['n_total'] = int(n_total)
    np.savez(os.path.join(APPENDIX_DIR, key + '.npz'), **payload)
    n_valid = int(z.size)
    n_sig   = int((z > sigma_thresh).sum())
    pct     = 100.0 * n_sig / n_valid
    tot_s   = f" of {int(n_total)}" if n_total is not None else ""
    print(f"\u2713 saved [{block} | {prompt}]  "
          f"n_sig={n_sig}, n_valid={n_valid}{tot_s}, ({pct:.1f}%), {pct/2.28:.1f}\u00d7 random")

# 현재 커널에 마지막 실행 결과가 남아 있으면 자동 저장 (보통 마지막 설정)
try:
    _n_total = int(J) if 'J' in globals() else (int(binary.shape[1]) if 'binary' in globals() else None)
    save_appendix_result(INJECT_BLOCK, PROMPT, z_all, SIGMA_THRESH, n_total=_n_total)
except NameError:
    print("\u26a0 메모리에 z_all/INJECT_BLOCK/PROMPT 없음 — "
          "각 설정 실행 직후 save_appendix_result(..., n_total=J) 호출")

In [ ]:
# # ════════════════════════════════════════════════════════════════════════════
# # (A) small-multiples 격자  +  (B) 요약 표   —  appendix_results/*.npz 자동 로드
# # ════════════════════════════════════════════════════════════════════════════
# import os, glob
# import numpy as np
# import matplotlib.pyplot as plt

# # ── Publication palette (matches other figures) ───────────────────────────────
# FIG_BG   = '#FFFFFF'
# GRID     = '#D5D9DE'
# SPINE    = '#3D3D3D'
# TEXT     = '#2A2A2A'
# MUTED    = '#666666'
# REFLINE  = '#8A8A8A'
# C_NULL   = '#A8A8A8'   # not significant
# C_SIG    = '#B85450'   # significant
# C_THRESH = '#2F4B7C'   # 2σ threshold
# C_RAND   = REFLINE
# C_HEADER = '#2F4B7C'   # table header
# ROW_ALT  = '#ECEFF3'   # alternating table row

# plt.rcParams.update({
#     'font.family': 'sans-serif',
#     'font.sans-serif': ['Arial', 'Helvetica Neue', 'DejaVu Sans'],
#     'font.size': 10,
#     'axes.labelcolor': TEXT,
#     'axes.edgecolor': SPINE,
#     'axes.linewidth': 0.8,
#     'axes.spines.top': False,
#     'axes.spines.right': False,
#     'grid.color': GRID,
#     'grid.alpha': 0.85,
#     'grid.linewidth': 0.6,
#     'xtick.color': MUTED,
#     'ytick.color': MUTED,
#     'text.color': TEXT,
#     'figure.facecolor': FIG_BG,
#     'axes.facecolor': FIG_BG,
# })

# EXPECTED_PCT = 2.28  # one-sided 2σ under N(0,1)

# recs = []
# for f in sorted(glob.glob(os.path.join(APPENDIX_DIR, '*.npz'))):
#     d   = np.load(f, allow_pickle=True)
#     z   = np.asarray(d['z_all'], dtype=float)
#     sig = float(d['sigma_thresh'])
#     ns  = int((z > sig).sum())
#     n_total = int(d['n_total']) if 'n_total' in d.files else None
#     recs.append(dict(block=str(d['block']), prompt=str(d['prompt']), z=z, sigma=sig,
#                      n_valid=int(z.size), n_sig=ns, n_total=n_total,
#                      pct=100.0*ns/z.size))
# assert recs, "appendix_results 가 비어 있습니다. 먼저 save_appendix_result(...)로 저장하세요."

# def _border(b):
#     return {'down_blocks':0, 'mid_block':1, 'up_blocks.0':2, 'up_blocks.1':3}.get(b, 99)
# def _short(p):
#     return p.replace('a photo of an ', '').replace('a photo of a ', '').strip() or p

# blocks  = sorted({r['block']  for r in recs}, key=_border)
# prompts = sorted({r['prompt'] for r in recs})
# lut     = {(r['block'], r['prompt']): r for r in recs}

# # ── (A) 격자: ranked z-score (왼쪽 패널만) ───────────────────────────────────
# nrows, ncols = len(prompts), len(blocks)
# figA, axes = plt.subplots(nrows, ncols, figsize=(3.4*ncols, 3.0*nrows), squeeze=False)
# for ri, prm in enumerate(prompts):
#     for ci, blk in enumerate(blocks):
#         ax = axes[ri][ci]
#         r  = lut.get((blk, prm))
#         if r is None:
#             ax.text(0.5, 0.5, '(pending)', ha='center', va='center', style='italic',
#                     fontsize=11, color=GRID, transform=ax.transAxes)
#             ax.set_xticks([]); ax.set_yticks([])
#             for s in ax.spines.values(): s.set_visible(False)
#         else:
#             z = np.sort(r['z']); ranks = np.arange(1, z.size+1); m = z > r['sigma']
#             ax.scatter(ranks[~m], z[~m], c=C_NULL, s=6, alpha=0.48, zorder=3, edgecolors='none')
#             ax.scatter(ranks[m],  z[m],  c=C_SIG,  s=6, alpha=0.88, zorder=4,
#                        edgecolors='white', linewidths=0.25)
#             ax.axhline(r['sigma'], color=C_THRESH, lw=1.1, ls=(0, (5, 4)), zorder=5)
#             ax.axhline(0, color=C_RAND, lw=0.8, ls=(0, (4, 4)), alpha=0.75, zorder=2)
#             tot_s = f" of {r['n_total']}" if r['n_total'] is not None else ""
#             ax.text(0.04, 0.96,
#                     f"{r['n_sig']}/{r['n_valid']}{tot_s}  ({r['pct']:.1f}%)\n{r['pct']/EXPECTED_PCT:.1f}× random",
#                     transform=ax.transAxes, ha='left', va='top',
#                     fontsize=8.0, fontweight='600', color=C_SIG)
#             ax.grid(True, color=GRID, alpha=0.85, lw=0.6)
#             ax.tick_params(axis='both', labelsize=8, colors=MUTED)
#             ax.spines['left'].set_color(SPINE)
#             ax.spines['bottom'].set_color(SPINE)
#         if ri == 0:        ax.set_title(blk, fontsize=10.5, fontweight='bold', pad=8, color=TEXT)
#         if ci == 0:        ax.set_ylabel(f'"{prm}"\n\nz-score', fontsize=9.5, color=TEXT)
#         if ri == nrows-1:  ax.set_xlabel('hyperplane rank', fontsize=9, color=TEXT)

# figA.tight_layout(rect=[0, 0, 1, 0.97])
# figA.savefig('appendix_grid_coherence.png', dpi=160, bbox_inches='tight', facecolor=FIG_BG)
# plt.show()
# print("saved: appendix_grid_coherence.png")

# # ── (B) 요약 표 ──────────────────────────────────────────────────────────────
# rows = []
# for blk in blocks:
#     for prm in prompts:
#         r = lut.get((blk, prm))
#         if r is not None:
#             valid_col = (f"{r['n_valid']}/{r['n_total']}" if r['n_total'] is not None
#                          else str(r['n_valid']))
#             rows.append((blk, _short(prm), valid_col, str(r['n_sig']),
#                          f"{r['pct']:.1f}%", f"{r['pct']/EXPECTED_PCT:.1f}×"))
# col_labels = ['Block', 'Prompt', 'Valid / Total', '>2σ', '% >2σ', 'vs. random']

# figT, axT = plt.subplots(figsize=(8.5, 0.6 + 0.5*len(rows)))
# axT.axis('off')
# tbl = axT.table(cellText=rows, colLabels=col_labels, loc='center', cellLoc='center')
# tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 1.5)
# for (rr, cc), cell in tbl.get_celld().items():
#     if rr == 0:
#         cell.set_facecolor(C_HEADER)
#         cell.set_text_props(color='white', fontweight='bold')
#     else:
#         cell.set_facecolor(ROW_ALT if rr % 2 else FIG_BG)
#         cell.set_text_props(color=TEXT)
#     cell.set_edgecolor(GRID)
# figT.suptitle('Summary: gate-neuron semantic coherence (>2σ enrichment)',
#               fontsize=11, fontweight='bold', color=TEXT)
# figT.savefig('appendix_summary_table.png', dpi=160, bbox_inches='tight', facecolor=FIG_BG)
# plt.show()
# print("saved: appendix_summary_table.png")

# # ── 논문용 LaTeX (booktabs) ──────────────────────────────────────────────────
# tex = [r"\begin{tabular}{llrrrr}", r"\toprule",
#        r"Block & Prompt & Valid / Total & $>2\sigma$ & \% $>2\sigma$ & vs.\ random \\",
#        r"\midrule"]
# for blk, prm, nv, ns, pct, ratio in rows:
#     pct_l   = pct.replace('%', r'\%')
#     ratio_l = ratio.replace('×', r'$\times$')
#     tex.append(f"{blk} & {prm} & {nv} & {ns} & {pct_l} & {ratio_l} " + r"\\")
# tex += [r"\bottomrule", r"\end{tabular}"]
# print("\n% ---- LaTeX (booktabs) ----")
# print("\n".join(tex))